In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

In [2]:
# Initialize Spark session
spark = SparkSession.builder.appName("JobStageTaskExample").getOrCreate()

# Disabling WholeStageCodegen to see the actual physical operators in the execution plan
spark.conf.set("spark.sql.codegen.wholeStage", "false")

# Disabling adaptive query execution to see the actual physical operators in the execution plan
spark.conf.set("spark.sql.adaptive.enabled", "false")

24/07/11 18:15:10 WARN Utils: Your hostname, paulo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
24/07/11 18:15:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/11 18:15:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Define schema
schema = StructType([
    StructField("date", StringType(), True),
    StructField("name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", DoubleType(), True)
])

In [56]:
# Sample data
data = [
    ("2024-07-01", "Alice", "Electronics", 1200.50),
    ("2024-07-02", "Bob", "Furniture", 850.00),
    ("2024-07-03", "Catherine", "Electronics", 399.99),
    ("2024-07-04", "Alice", "Furniture", 450.00),
    ("2024-07-05", "Bob", "Electronics", 650.75)
]

df = spark.createDataFrame(data, schema)
df.write.mode("overwrite").parquet("./tmp/customers")

In [8]:
customers_df.show(5)

+----------+---------+-----------+------+
|      date|     name|   category|amount|
+----------+---------+-----------+------+
|2024-07-03|Catherine|Electronics|399.99|
|2024-07-01|    Alice|Electronics|1200.5|
|2024-07-04|    Alice|  Furniture| 450.0|
|2024-07-05|      Bob|Electronics|650.75|
|2024-07-02|      Bob|  Furniture| 850.0|
+----------+---------+-----------+------+



In [4]:
# Provide schema explicitly to avoid unecessary job
customers_df = spark.read.schema(schema).parquet("./tmp/customers")

In [5]:
df_filtered = customers_df.filter(customers_df["Amount"] > 500)
category_sales = df_filtered.groupBy("category").sum("Amount")

result = category_sales.collect()

In [31]:
print(f"Total sales per customer: {result}")

24/06/30 22:33:52 WARN TaskSetManager: Stage 0 contains a task of very large size (1282 KiB). The maximum recommended task size is 1000 KiB.


Execution time with default partitions (200): 5.898292541503906 seconds


In [62]:
# Stop the Spark session
spark.stop()